# Blackboard DEMO

## Setup

In [ ]:
from src.board import Board

question = """
Диагностика экономического и пространственного состояния г. Тюмень:
- социально-экономический анализ (демография, производительность труда, структура и динамика ВГП, рынок
- труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность);
- пространственный анализ (функциональная организация территории, структура землевладения, природноэкологический каркас, архитектурно-градостроительная среда);
- транспортный анализ (городской и внешний пассажирский и грузовой транспорт, парковки, пешеходные зоны);
- анализ инженерной инфраструктуры (водоснабжение и водоотведение, энергоснабжение, теплоснабжение).
"""

board = Board(question=question)

In [3]:
from src.agents.factories import *

generator_agent = create_generator_agent(board)
roles = generator_agent.invoke([]).roles

In [4]:
expert_agents = [create_expert_agent(r, board) for r in roles]

for expert in expert_agents:
    print(str.join(', ', expert.info.values()))

1bb43d, Градостроительный планировщик, Эксперт в области стратегического развития территорий, умеющий формировать долгосрочные цели города, учитывая пространственное планирование,
8f24bc, Экономист регионального развития, Специалист, анализирующий экономические тенденции, инвестиционный потенциал и социально‑экономические приоритеты региона, формируя цели, 
06e6c7, Стратегический консультант по публичной политике, Профессионал в разработке и согласовании муниципальных стратегий, обеспечивающий согласованность целей с интересами граждан, бизнес‑сообща­


In [5]:
# from src.searcher import WikipediaSearcher
# wikipedia_searcher = WikipediaSearcher()

planner_agent = create_planner_agent(board)
# wikipedia_agent = create_wikipedia_agent(wikipedia_searcher, board)
critic_agent = create_critic_agent(board)
cleaner_agent = create_cleaner_agent(board)
decider_agent = create_decider_agent(board)

role_agents = [planner_agent, *expert_agents, critic_agent, cleaner_agent, decider_agent] # wikipedia_agent, 
controller_agent = create_controller_agent(role_agents, board)

## Experiment

In [6]:
from src.agents import Agent

def _get_agent(agent_id : str, agents : list[Agent]) -> Agent | None:
    for agent in agents:
        if agent.id == agent_id:
            return agent
    return None

def get_agents(agents_ids : list[str], agents : list[Agent]):
    result = []
    for agent_id in agents_ids:
        agent = _get_agent(agent_id, agents)
        if agent is not None:
            result.append(agent)
    return result

In [ ]:
from tqdm import tqdm
from langchain.messages import AIMessage, HumanMessage, SystemMessage

is_final = False

def get_invoke_message() -> SystemMessage:
    return SystemMessage(f"Записей на доске: {len(board.notes)}")

for i in range(1):
    print(f'Итерация {i+1}')
    agents_ids = controller_agent.invoke([get_invoke_message()], force=True).agents_ids
    agents = get_agents(agents_ids, role_agents)
    print(str.join(', ', [a.role.name for a in agents]))

    for a in tqdm(agents):
        try:
            response = a.invoke([get_invoke_message()], force=False)
        except:
            print(a.role.name + ' перегрелся')
            continue

        if a == decider_agent: # DECIDER AGENT
            note = response.note
            is_final = response.is_final
        elif a == cleaner_agent: # CLEANER AGENT
            notes_ids = response.notes_ids
            board.remove_notes(notes_ids)
            continue
        else:
            note = response

        board.add_note(note.content, a.id, a.role.name)

        if is_final:
            break

    if is_final:
        break

Итерация 1
Планировщик, Градостроительный планировщик, Экономист регионального развития, Стратегический консультант по публичной политике, Критик, Уборщик, Арбитр


 86%|████████▌ | 6/7 [02:49<00:26, 26.40s/it]

Уборщик перегрелся


100%|██████████| 7/7 [03:02<00:00, 26.10s/it]

Арбитр перегрелся


In [10]:
board.print()

╭─────────────────────────────── 📌#f72daa by 1200e2 (Планировщик) ────────────────────────────────╮
│ ### Описание задачи                                                                              │
│ Разработать документ «Целеполагание города Гатчина (Ленинградская область) на 2026‑2035 годы», в │
│ котором будут сформулированы стратегические направления развития, приоритетные задачи и основные │
│ идеи, но без конкретных количественных целевых показателей (KPIs, индикаторов). Документ должен  │
│ стать базой для дальнейшего стратегического планирования и последующего уточнения целей с        │
│ метриками.                                                                                       │
│                                                                                                  │
│ ### План решения задачи (шаги)                                                                   │
│ 1. **Формирование рабочей группы**                                                               │
│    - Определить ключевых участников: представители администрации города, планово‑экономического  │
│ отдела, профильных министерств (жилищно‑коммунальное хозяйство, транспорт, культура, спорт,      │
│ экология), эксперты‑аналитики, представители бизнеса и гражданского общества.                    │
│    - Назначить координатора (ответственного за сбор и согласование материалов).                  │
│                                                                                                  │
│ 2. **Сбор исходных данных и контекста**                                                          │
│    - Свести в один файл актуальную информацию о текущем состоянии города (демография, экономика, │
│ инфраструктура, социальные услуги, экологическая ситуация, бюджетные ресурсы).                   │
│    - Проанализировать уже существующие стратегии и программы развития (например, Стратегия       │
│ развития муниципального образования Гатчина, программы «Развитие территории», «Экологический     │
│ план» и т.п.).                                                                                   │
│    - Выявить внешние факторы (региональная политика, федеральные программы, демографические      │
│ тенденции, климатические изменения).                                                             │
│                                                                                                  │
│ 3. **Определение миссии и видения города**                                                       │
│    - Провести воркшоп с участниками группы для формулирования миссии (зачем существует город) и  │
│ видения (каким он будет в 2035 году).                                                            │
│    - Зафиксировать формулировки в виде коротких, понятных предложений, которые будут             │
│ использоваться в дальнейшем документе.                                                           │
│                                                                                                  │
│ 4. **Идентификация стратегических направлений**                                                  │
│    - На основе анализа данных и сформулированного видения определить 4‑6 основных стратегических │
│ направлений (например: «Устойчивое развитие городской инфраструктуры», «Повышение качества жизни │
│ населения», «Экономическое diversification», «Экологическая безопасность», «Развитие культурного │
│ и спортивного потенциала», «Грамотное управление и цифровизация»).                               │
│    - Для каждого направления сформулировать ключевые приоритетные задачи (без указания           │
│ количественных целей). Пример: «Модернизация транспортной сети», «Развитие доступного жилья»,    │
│ «Сокращение выбросов загрязняющих веществ», «Создание условий для привлечения инвестиций в       │
│ инновационные отрасли».                                                                          │
│                                                             

╭────────────────────── 📌#af7fd9 by 1bb43d (Градостроительный планировщик) ───────────────────────╮
│ ### Дополнения к плану разработки целеполагания г. Гатчина 2026‑2035 гг.                         │
│                                                                                                  │
│ **1. Расширить перечень стратегических направлений**                                             │
│ Помимо предложенных 4‑6 направлений, рекомендую включить два дополнительных, критически важных   │
│ для нашего региона:                                                                              │
│ - **Климатическая адаптация и устойчивость** – подготовка к повышению уровня грунтовых вод,      │
│ экстремальным температурным режимам, развитие «зелёных» инфраструктурных решений (зеленые крыши, │
│ дождеприёмные системы, озеленение улиц).                                                         │
│ - **Цифровая трансформация и умный город** – создание цифрового двойника территории, открытых    │
│ данных о городской среде, внедрение систем мониторинга качества воздуха, энергопотребления и     │
│ транспортных потоков.                                                                            │
│                                                                                                  │
│ **2. Принципы реализации – более детализировать**                                                │
│ | Принцип | Краткое описание |                                                                   │
│ |---|---|                                                                                        │
│ | Интеграция с региональными программами | Согласование планов с «Стратегией развития            │
│ Ленинградской области», федеральными программами «Развитие транспортной инфраструктуры»,         │
│ «Экология‑2030» и др. |                                                                          │
│ | Публичное участие и открытость | Публичные онлайн‑консультации, интерактивные карты            │
│ предложений, участие в формировании приоритетов через платформу «Голос Гатчины». |               │
│ | Гибкость и адаптивность | Периодический пересмотр приоритетных задач (каждые 2‑3 года) с       │
│ учётом новых данных и изменений внешней среды. |                                                 │
│ | Финансовая устойчивость | Привлечение инвестиций через публично‑частные партнёрства, гранты    │
│ Фонда «Развитие регионов», муниципальные облигации. |                                            │
│ | Ориентированность на качество жизни | Оценка влияния каждой задачи на индексы качества жизни   │
│ (доступность услуг, безопасность, экологию). |                                                   │
│                                                                                                  │
│ **3. Предлагаемый график работ (2024‑2025 гг.)**                                                 │
│ | Этап | Срок | Ответственный | Ключевые результаты |                                            │
│ |---|---|---|---|                                                                                │
│ | Формирование рабочей группы и назначение координатора | 1‑й квартал 2024 | Глава администрации │
│ (координатор) | Список участников, регламенты встреч |                                           │
│ | Сбор и систематизация исходных данных | 2‑й‑3‑й кварталы 2024 | Планово‑экономический отдел |  │
│ Единый аналитический досье (демография, экономика, инфраструктура) |                             │
│ | Воркшопы по миссии, видению и стратегическим направлениям | 4‑й квартал 2024 | Координатор +   │
│ внешние фасилитаторы | Формулировки миссии, видения, перечень направлений |                      │
│ | Разработка чернового варианта документа | 1‑й‑2‑й кварталы 2025 | Рабочая группа (разделы      │
│ распределены) | Черновой документ в репозитории |                                                │
│ | Экспертная проверка и публичные консультации | 3‑й квартал

╭───────────────────── 📌#57ab95 by 8f24bc (Экономист регионального развития) ─────────────────────╮
│ ### Предложения экономиста регионального развития (id 8f24bc)                                    │
│                                                                                                  │
│ #### 1. Дополнение к стратегическим направлениям – экономический фокус                           │
│ | № | Наименование направления (расширенный) | Краткое обоснование | Ключевые приоритетные       │
│ задачи (без KPI) |                                                                               │
│ |---|----------------------------------------|--------------------|----------------------------- │
│ -----------|                                                                                     │
│ | 1 | **Устойчивое развитие городской инфраструктуры** | Обеспечение надежных и                  │
│ энерго‑эффективных сетей, подготовка к климатическим рискам. | • Модернизация коммунальных сетей │
│ (водоснабжение, теплоснабжение, электросети) с применением «умных» датчиков.<br>• Внедрение      │
│ систем управления энергопотреблением в муниципальных зданиях.<br>• Развитие «зеленой»            │
│ инфраструктуры (зеленые крыши, дождеприёмные системы, озеленение улиц). |                        │
│ | 2 | **Повышение качества жизни населения** | Сбалансированное развитие социальной сферы,       │
│ доступность услуг, безопасность. | • Расширение сети доступного жилья (модульные и               │
│ энерго‑эффективные дома).<br>• Улучшение медицинского и образовательного обслуживания (цифровые  │
│ сервисы, телемедицина).<br>• Развитие городской среды для активного образа жизни (парки,         │
│ велосипедные дорожки, спортивные комплексы). |                                                   │
│ | 3 | **Экономическое diversification и инновационный кластер** | Снижение зависимости от        │
│ традиционных отраслей, привлечение новых инвестиций, создание рабочих мест. | • Формирование     │
│ «инновационного парка» в зоне свободного экономического развития (технопарк, коворкинги,         │
│ акселераторы).<br>• Поддержка малых и средних предприятий (гранты, субсидии, упрощённые          │
│ лицензии).<br>• Привлечение инвестиций в отрасли: логистика, IT‑услуги, биотехнологии, лёгкое    │
│ производство, туризм. |                                                                          │
│ | 4 | **Экологическая безопасность и климатическая адаптация** | Учет растущих климатических     │
│ рисков, снижение загрязнения, сохранение природных ресурсов. | • Сокращение выбросов             │
│ загрязняющих веществ (переход на электромобили, модернизация автопарка муниципального            │
│ транспорта).<br>• Развитие системы мониторинга качества воздуха и водных ресурсов (цифровой      │
│ двойник).<br>• План действий по адаптации к повышению уровня грунтовых вод и экстремальным       │
│ температурам. |                                                                                  │
│ | 5 | **Цифровая трансформация и умный город** | Повышение эффективности управления, открытость  │
│ данных, улучшение сервисов для жителей. | • Создание цифрового двойника территории               │
│ (моделирование транспортных потоков, энергопотребления, экологических параметров).<br>• Открытая │
│ платформа данных (Open Data) для бизнеса и граждан.<br>• Внедрение систем «умного» управления    │
│ уличным освещением, парковкой, сбором мусора. |                                                  │
│ | 6 | **Грамотное управление, финансовая устойчивость и публичное участие** | Прозрачность,      │
│ эффективность расходов, привлечение внешних ресурсов. | • Интеграция планов с региональными и    │
│ федеральными программами (Стратегия развития Ленобласти, «Экология‑2030», «Развитие транспортной │
│ инфраструктуры»).<br>• Развитие механизмов публично‑частного партнёрства (PPP), муниципальных    │
│ облигаций, грантов Фонда «Развитие регионов».<br>• Платформа

╭───────────── 📌#cacdbc by 06e6c7 (Стратегический консультант по публичной политике) ─────────────╮
│ ### Предложения стратегического консультанта по публичной политике (id 06e6c7)                   │
│                                                                                                  │
│ #### 1. Общий подход к формулированию целеполагания без KPI                                      │
│ | Принцип | Как реализовать |                                                                    │
│ |---|---|                                                                                        │
│ | **SMART‑без‑KPI** (Specific, Qualitative‑Measurable, Achievable, Relevant, Time‑bound) | Для   │
│ каждой приоритетной задачи указываем **качественный** критерий измеримости (например, *«повысить │
│ долю энерго‑эффективных зданий в муниципальном парке»*), а также **ориентировочный срок**        │
│ (2026‑2030). Это позволит сразу перейти к этапу метризации без доработки формулировок. |         │
│ | **Интеграция с бюджетным планированием** | В каждом стратегическом направлении указываем,      │
│ какие бюджетные статьи (операционные, капитальные, субвенции) могут быть задействованы. Это      │
│ создаст основу для последующего согласования с финансовой программой города. |                   │
│ | **Гибкость и адаптивность** | В документе фиксируем **механизм пересмотра** приоритетных задач │
│ каждые 2‑3 года (публичный отчёт, обновление сценариев). |                                       │
│ | **Публичность и открытость** | Все формулировки публикуются в открытом репозитории             │
│ (Confluence/Google Docs) и сопровождаются интерактивной картой «Приоритетные задачи» в платформе │
│ «Голос Гатчины». |                                                                               │
│ | **Управление рисками** | Для каждой задачи описываем **ключевые риски** (финансовые,           │
│ регуляторные, климатические) и **мягкие меры** их снижения (например, резервные источники        │
│ финансирования, альтернативные технологии). |                                                    │
│                                                                                                  │
│ #### 2. Структура будущего документа (расширенная)                                               │
│ 1. **Введение** – цель, методология (SMART‑без‑KPI, сценарный анализ), сроки разработки.         │
│ 2. **Миссия и видение** – коротко, лаконично, отражающие уникальность Гатчины (пример: *«Гатчина │
│ – современный, экологически безопасный и инновационный центр культурного и экономического        │
│ развития Ленинградской области»*).                                                               │
│ 3. **Стратегические направления** – 6‑8 направлений (см. ниже) с обоснованием их взаимосвязи и   │
│ вклада в миссию.                                                                                 │
│ 4. **Приоритетные задачи** – по каждому направлению 3‑5 задач, сформулированных по               │
│ SMART‑без‑KPI, с указанием **временных рамок** и **ответственных органов**.                      │
│ 5. **Принципы реализации** – интеграция с региональными программами, публичное участие,          │
│ гибкость, финансовая устойчивость, ориентация на качество жизни (таблица, как в дополнении       │
│ планировщика).                                                                                   │
│ 6. **Механизм мониторинга и контроля** – описание будущего этапа метризации (рабочая группа      │
│ «Метрики и мониторинг»), инструменты (GIS‑модели, Power BI‑дашборды), периодичность отчётности.  │
│ 7. **Риски и меры их снижения** – таблица рисков по направлениям.                                │
│ 8. **Приложения** – аналитический досье, список участников, ссылки на нормативные документы,     │
│ шаблоны «Концептуальная задача».                                                                 │
│                                                             

╭────────────────────────────────── 📌#316df4 by 7d5c4d (Критик) ──────────────────────────────────╮
│ **Анализ записей на общей доске**                                                                │
│                                                                                                  │
│ | Запись | Автор | Краткое содержание | Оценка полезности |                                      │
│ |--------|-------|--------------------|-------------------|                                      │
│ | 1 | Планировщик (f72daa) | Общая структура целеполагания — стратегические направления,         │
│ приоритетные задачи, этапы реализации, механизмы контроля и мониторинга. | **Полезна** – задаёт  │
│ фундаментальную схему, в которой впоследствии уточняются детали. |                               │
│ | 2 | Градостроительный планировщик (af7fd9) | Конкретные градостроительные инициативы (жильё,   │
│ транспорт, инфраструктура, экология, цифровизация) с указанием сроков, ответственных и ожидаемых │
│ результатов. | **Полезна** – наполняет общую схему конкретными проектами, привязывает их к       │
│ срокам и исполнителям. |                                                                         │
│ | 3 | Экономист (57ab95) | Экономическое обоснование инициатив, оценка финансовых ресурсов,      │
│ источники финансирования, модели доходов/расходов, риски и их смягчение. | **Полезна** –         │
│ обеспечивает финансовую целесообразность и устойчивость предложенных проектов. |                 │
│ | 4 | Стратегический консультант (cacdbc) | Расширенный набор стратегических направлений,        │
│ детализированные задачи, KPI‑подход (без конкретных числовых целей), рекомендации по управлению, │
│ мониторингу и коммуникации. | **Полезна** – дополняет и уточняет стратегию, вводит практические  │
│ инструменты управления и контроля. |                                                             │
│                                                                                                  │
│ ### Вывод                                                                                        │
│                                                                                                  │
│ Все четыре записи вносят **уникальный и необходимый вклад** в формирование целеполагания города  │
│ Гатчина на 2026‑2035 годы:                                                                       │
│ - первая запись задаёт общую структуру;                                                          │
│ - вторая – конкретизирует градостроительные проекты;                                             │
│ - третья – подкрепляет их экономическим обоснованием;                                            │
│ - четвёртая – расширяет стратегический контекст и предлагает инструменты реализации.             │
│                                                                                                  │
│ Никаких **бесполезных** или **избыточных** записей не обнаружено. Каждая из них покрывает        │
│ отдельный аспект (структура, проекты, финансы, управление) и вместе образует целостный набор     │
│ материалов, необходимых для дальнейшей работы.                                                   │
│                                                                                                  │
│ **Дальнейшие шаги:** ожидаю дополнительной информации (например, требования к формату итогового  │
│ документа, приоритеты заказчика, ограничения по бюджету), чтобы приступить к синтезу всех        │
│ материалов в единый план без целевых показателей.                                                │
│                                                                                                  │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
role_agents[2].messages

[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nDevelop a comprehensive strategic plan for the city of Gatchina (Leningrad Oblast) for 2026‑2035, incorporating existing urban‑planning directions and a newly added economic development block, and be ready to provide further analysis, prioritization, implementation planning, or presentation materials as requested.\n\n## SUMMARY\n- **Existing strategic directions (7 points, Note ID\u202f905e2f, authored by “Градостроительный планировщик”):**  \n  1. Sustainable development & environmental safety – green transport, urban greening, eco‑infrastructure.  \n  2. Transport & logistics integration – multimodal hub, bike lanes, traffic management.  \n  3. Housing & communal development – new districts, renovation, social infrastructure.  \n  4. Economic development & diversification – innovation clusters, SME support, tourism, agri‑food.  \n  5. Digitalization & smart city – city platform, broadband, dat

In [11]:
board.print()

╭────────────────────── 📌#905e2f by a9f900 (Градостроительный планировщик) ───────────────────────╮
│ Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы (без количественных           │
│ показателей)                                                                                     │
│                                                                                                  │
│ 1. **Устойчивое развитие и экологическая безопасность**                                          │
│    - Формирование городской среды, ориентированной на снижение углеродного следа: развитие сети  │
│ электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка         │
│ муниципальных и частных организаций на экологически чистый транспорт.                            │
│    - Увеличение доли зеленых насаждений и открытых пространств: создание новых парков,           │
│ рекреационных зон вдоль реки Ижора, развитие линейных зеленых коридоров, интеграция              │
│ вертикального озеленения в архитектуре.                                                          │
│    - Внедрение принципов «зеленой» инфраструктуры в новых районах: энергоэффективные здания,     │
│ системы сбора и повторного использования дождевой воды, умные системы уличного освещения.        │
│                                                                                                  │
│ 2. **Транспортная и логистическая интеграция**                                                   │
│    - Развитие мультимодального транспортного узла, связывающего железнодорожный вокзал,          │
│ автовокзал и будущие линии скоростного общественного транспорта, обеспечивая удобную связь с     │
│ Санкт-Петербургом и другими крупными центрами региона.                                           │
│    - Расширение сети велосипедных дорожек и создание безопасных маршрутов для пешеходов,         │
│ интегрированных в жилые кварталы и зоны отдыха.                                                  │
│    - Оптимизация дорожной сети с учётом прогнозируемого роста автотранспорта, внедрение          │
│ интеллектуальных систем управления движением.                                                    │
│                                                                                                  │
│ 3. **Жилищно‑коммунальное развитие**                                                             │
│    - Планирование новых жилых районов с разнообразным типовым фондом (многоэтажные,              │
│ среднеэтажные, таунхаусы) для разных социальных групп, с акцентом на доступность и качество      │
│ инфраструктуры.                                                                                  │
│    - Реновация и модернизация существующего жилого фонда, улучшение энергоэффективности,         │
│ внедрение систем «умный дом» в муниципальном жилье.                                              │
│    - Обеспечение равномерного доступа к социальным объектам (школы, детские сады, поликлиники) в │
│ новых и реконструируемых районах.                                                                │
│                                                                                                  │
│ 4. **Экономическое развитие и диверсификация**                                                   │
│    - Создание инновационных кластеров в сфере IT, креативных индустрий и высокотехнологичного    │
│ производства, привлечение инвестиций через специальные экономические зоны и бизнес‑инкубаторы.   │
│    - Поддержка малого и среднего предпринимательства, развитие ремесленных и туристических       │
│ направлений, связанных с историческим наследием Гатчины (Гатчинский дворец, музеи, культурные    │
│ мероприятия).                                                                                    │
│    - Развитие агропродовольственного сектора в окрестностях города, поддержка фермерских рынков  │
│ и локальных брендов.                                        

╭───────────────────── 📌#413697 by 531719 (Экономист регионального развития) ─────────────────────╮
│ Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы (экономический блок, без      │
│ количественных показателей)                                                                      │
│                                                                                                  │
│ **1. Диверсификация экономической структуры**                                                    │
│ - Формирование мультиотраслевого профиля города, где традиционные отрасли (производство,         │
│ сельское хозяйство) дополняются новыми секторами: IT‑технологии, креативные индустрии,           │
│ биотехнологии, экологическое производство.                                                       │
│ - Поддержка создания инновационных кластеров в сотрудничестве с университетами и                 │
│ исследовательскими центрами Санкт‑Петербурга, привлечение компаний‑стартапов через специальные   │
│ технопарки и бизнес‑инкубаторы.                                                                  │
│                                                                                                  │
│ **2. Инвестиционный климат и финансовая поддержка**                                              │
│ - Разработка пакета льготных условий для инвесторов: упрощённые процедуры получения разрешений,  │
│ налоговые послабления в первые годы эксплуатации, гарантии доступа к муниципальной земле под     │
│ развитие инфраструктуры.                                                                         │
│ - Создание муниципального фонда развития малого и среднего бизнеса, предоставляющего субсидии,   │
│ гранты и гарантии по кредитам для локальных предпринимателей, особенно в сфере ремесел, туризма  │
│ и агропродовольствия.                                                                            │
│                                                                                                  │
│ **3. Развитие человеческого капитала**                                                           │
│ - Инвестиции в образование и профессиональную подготовку, ориентированные на потребности новых   │
│ отраслей: программы переподготовки, совместные курсы с ведущими вузами, развитие центров         │
│ повышения квалификации.                                                                          │
│ - Привлечение и удержание талантливой молодёжи через создание комфортных условий жизни (жильё,   │
│ досуг, культурные мероприятия) и карьерных возможностей в инновационных проектах.                │
│                                                                                                  │
│ **4. Экономика знаний и цифровая трансформация**                                                 │
│ - Внедрение единой цифровой платформы для взаимодействия бизнеса, муниципальных служб и граждан, │
│ позволяющей автоматизировать процессы лицензирования, подачи заявок на гранты и мониторинга      │
│ инвестиций.                                                                                      │
│ - Поддержка широкополосного доступа и развитие цифровой инфраструктуры, обеспечивая основу для   │
│ удалённой работы, электронных сервисов и онлайн‑образования.                                     │
│                                                                                                  │
│ **5. Устойчивое производство и «зелёная» экономика**                                             │
│ - Стимулирование перехода предприятий к энерго‑ и ресурсосберегающим технологиям, внедрение      │
│ систем замкнутого цикла, поддержка проектов по использованию возобновляемой энергии (солярные и  │
│ ветровые установки).                                                                             │
│ - Развитие агропродовольственного сектора с акцентом на экологически чистое производство,        │
│ поддержка фермерских кооперативов, создание локальных брендо

╭───────────── 📌#da51e2 by 3d3be4 (Стратегический консультант по публичной политике) ─────────────╮
│ Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы (без количественных           │
│ показателей)                                                                                     │
│                                                                                                  │
│ **1. Устойчивое развитие и экологическая безопасность**                                          │
│ - Формировать городскую среду, ориентированную на снижение углеродного следа: развитие сети      │
│ электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка         │
│ муниципальных и частных организаций на экологически чистый транспорт.                            │
│ - Увеличивать долю зеленых насаждений и открытых пространств: создание новых парков,             │
│ рекреационных зон вдоль реки Ижора, развитие линейных зеленых коридоров, интеграция              │
│ вертикального озеленения в архитектуре.                                                          │
│ - Внедрять принципы «зеленой» инфраструктуры в новых районах: энергоэффективные здания, системы  │
│ сбора и повторного использования дождевой воды, умные системы уличного освещения.                │
│                                                                                                  │
│ **2. Транспортная и логистическая интеграция**                                                   │
│ - Развивать мультимодальный транспортный узел, связывающий железнодорожный вокзал, автовокзал и  │
│ будущие линии скоростного общественного транспорта, обеспечивая удобную связь с                  │
│ Санкт‑Петербургом и другими крупными центрами региона.                                           │
│ - Расширять сеть велосипедных дорожек и создавать безопасные маршруты для пешеходов,             │
│ интегрированные в жилые кварталы и зоны отдыха.                                                  │
│ - Оптимизировать дорожную сеть с учётом прогнозируемого роста автотранспорта, внедрять           │
│ интеллектуальные системы управления движением.                                                   │
│                                                                                                  │
│ **3. Жилищно‑коммунальное развитие**                                                             │
│ - Планировать новые жилые районы с разнообразным типовым фондом (многоэтажные, среднеэтажные,    │
│ таунхаусы) для разных социальных групп, с акцентом на доступность и качество инфраструктуры.     │
│ - Реновировать и модернизировать существующий жилой фонд, повышать энергоэффективность, внедрять │
│ системы «умный дом» в муниципальном жилье.                                                       │
│ - Обеспечивать равномерный доступ к социальным объектам (школы, детские сады, поликлиники) в     │
│ новых и реконструируемых районах.                                                                │
│                                                                                                  │
│ **4. Экономическое развитие и диверсификация**                                                   │
│ - Формировать мультиотраслевой профиль города, где традиционные отрасли (производство, сельское  │
│ хозяйство) дополняются новыми секторами: IT‑технологии, креативные индустрии, биотехнологии,     │
│ экологическое производство.                                                                      │
│ - Создавать инновационные кластеры в сотрудничестве с университетами и исследовательскими        │
│ центрами Санкт‑Петербурга, привлекать стартапы через технопарки и бизнес‑инкубаторы.             │
│ - Поддерживать малый и средний бизнес, развивать ремесленные и туристические направления,        │
│ связанные с историческим наследием Гатчины (Гатчинский дворец, музеи, культурные мероприятия).   │
│ - Развивать агропродовольственный сектор с акцентом на эколо

╭───────────────────── 📌#d33d0a by 531719 (Экономист регионального развития) ─────────────────────╮
│ 1. Диверсификация экономической структуры – развитие новых отраслей (IT, креатив, биотех,        │
│ «зелёное» производство) и поддержка инновационных кластеров, технопарков, сотрудничество с       │
│ университетами Санкт‑Петербурга.                                                                 │
│ 2. Инвестиционный климат и финансовая поддержка – налоговые льготы, упрощённые процедуры         │
│ получения разрешений, создание муниципального фонда поддержки МСБ с субсидиями, грантами и       │
│ гарантиями кредитов.                                                                             │
│ 3. Развитие человеческого капитала – программы профессионального обучения и переподготовки,      │
│ совместные проекты с вузами и исследовательскими центрами, привлечение и удержание талантов      │
│ через улучшение качества жизни.                                                                  │
│ 4. Экономика знаний и цифровая трансформация – единая цифровая платформа для взаимодействия      │
│ бизнеса и муниципалитета, расширение широкополосного доступа, поддержка удалённой работы и       │
│ электронных государственных услуг.                                                               │
│ 5. Устойчивое производство и «зелёная» экономика – внедрение энерго‑ и ресурсосберегающих        │
│ технологий, развитие циркулярной экономики, проекты в области возобновляемой энергии,            │
│ экологически чистого агропродовольственного сектора, локальный брендинг.                         │
│ 6. Интеграция в региональные и международные цепочки создания стоимости – модернизация           │
│ мультимодального транспортно‑логистического узла, участие в региональных программах, доступ к    │
│ федеральным и международным фондам, межмуниципальное сотрудничество.                             │
│ 7. Социально‑экономическая инклюзивность – равный доступ к экономическим возможностям для людей  │
│ с ограниченными возможностями, пожилых, семей; целевые программы поддержки; система социальной   │
│ защиты в переходные периоды.                                                                     │
│ 8. Городская резильентность и готовность к экономическим шокам – резервные финансовые механизмы, │
│ диверсификация муниципальных доходов, бюджетные буферы для кризисных ситуаций.                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:
role_agents[0].messages

[SystemMessage(content='Текущие записи на доске:\n[]', additional_kwargs={}, response_metadata={}, id='94d3928f-95ab-4c50-8abd-90599ec3f766'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 352, 'total_tokens': 385, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-oss:120b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-652', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d3efa-cb04-7412-ab23-081a648644df-0', tool_calls=[{'name': 'get_notes', 'args': {}, 'id': 'call_86hhpxh5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 352, 'output_tokens': 33, 'total_tokens': 385, 'input_token_details': {}, 'output_token_details': {}}),
 ToolMessage(content=[], name='get_notes', id='f3c36e4b-db68-4b63-80de-2de5a633b79f', tool_call_id='call_86hhpxh5'),
 AIMessage(content='', additional_kwargs=

In [12]:
board.notes

[Note(id='905e2f', author_id='a9f900', author_role='Градостроительный планировщик', content='Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы (без количественных показателей)\n\n1. **Устойчивое развитие и экологическая безопасность**\n   - Формирование городской среды, ориентированной на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка муниципальных и частных организаций на экологически чистый транспорт.\n   - Увеличение доли зеленых насаждений и открытых пространств: создание новых парков, рекреационных зон вдоль реки Ижора, развитие линейных зеленых коридоров, интеграция вертикального озеленения в архитектуре.\n   - Внедрение принципов «зеленой» инфраструктуры в новых районах: энергоэффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.\n\n2. **Транспортная и логистическая интеграция**\n   - Развитие мультимодального транспортного 

In [11]:
controller_agent.messages

[SystemMessage(content='Записи на доске:\n[]', additional_kwargs={}, response_metadata={}, id='067b7588-398e-4458-8b0f-94f92d9c6db4'),
 AIMessage(content='{\n  "agents_ids": [\n    "0a937f", "9ac9ae", "1c97cf", "a90919", "05c122", "7794f8"\n  ]\n}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 757, 'total_tokens': 805, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-oss:120b', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-85', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3ee7-00e6-7211-aa07-fda4552f8a7b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 757, 'output_tokens': 48, 'total_tokens': 805, 'input_token_details': {}, 'output_token_details': {}}),
 SystemMessage(content="Записи на доске:\n[{'id': 'f7e295', 'author_id': '0a937f', 'author_role': 'Планировщик', 'preview': ''}, {'id': 'f4a40

### Summarize

In [13]:
summarizer_actor_agent = Agent(
    tools=board.get_ro_tools(),  # оставляем инструменты
    system_prompt="""
    Ваша задача: синтезировать финальный ответ на поставленную задачу, основываясь ТОЛЬКО на материалах доски.
    
    Поставленная задача:
    `{question}`
    
    Алгоритм:
    1. Вызовите get_notes(), чтобы увидеть все записи
    2. Вызовите get_note() для КАЖДОЙ из них
    3. Синтезируйте финальный ответ
    4. Верните ТОЛЬКО финальный ответ, без лишних комментариев
    """.format(question=board.question)
)

summarizer_critic_agent = Agent(
    tools=board.get_ro_tools(),  # оставляем инструменты
    system_prompt="""
    Ваша задача: на основе черновика финального ответа на поставленную задачу и материалов на доске выдайте список замечаний.
    
    Поставленная задача:
    `{question}`
    """.format(question=board.question)
)

In [14]:
messages = []

n_gen = 3

for i in tqdm(range(n_gen)):
    actor_res = summarizer_actor_agent.invoke([HumanMessage(m) for m in messages[-1:]])
    messages.append(actor_res)

    if (i+1) < n_gen:
        critic_res = summarizer_critic_agent.invoke([HumanMessage(m) for m in messages[-1:]])
        messages.append(critic_res)

100%|██████████| 3/3 [03:41<00:00, 73.76s/it]


In [18]:
messages

['**Целеполагание г. Гатчина (Ленинградская область)\u202fна 2026‑2035\u202fгоды (без количественных показателей)**  \n\n---\n\n### 1. Устойчивое развитие и экологическая безопасность  \n- Формировать городскую среду, ориентированную на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, переход автопарка муниципальных и частных организаций на экологически чистый транспорт.  \n- Увеличивать долю зелёных насаждений и открытых пространств: создание новых парков и рекреационных зон вдоль реки Ижоры, развитие линейных зелёных коридоров, интеграция вертикального озеленения в архитектуру.  \n- Внедрять принципы «зелёной» инфраструктуры в новых районах: энергоэффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.  \n\n### 2. Транспортная и логистическая интеграция  \n- Создать мультимодальный транспортный узел, соединяющий железнодорожный вокзал, автовокзал и будущие линии скоростного обще

In [16]:
print(messages[-1])

**Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы  
(без количественных показателей)**  

---

### 1. Устойчивое развитие и экологическая безопасность  
- **Зелёная инфраструктура** – расширение сети парков, рекреационных зон вдоль реки Ижоры; создание линейных зелёных коридоров и вертикального озеленения в новых кварталах.  
- **Возобновляемая энергетика и энергетическая независимость** – стимулирование установки солнечных панелей, ветровых турбин и биогазовых установок; развитие муниципальных микросетей; поддержка «зелёных тарифов» и субсидий для частных владельцев ВИЭ.  
- **Энерго‑ и ресурсосбережение** – внедрение энергоэффективных технологий в зданиях, умного освещения и автоматизированного контроля потребления.  
- **Водоснабжение и качество воды** – интеграция систем сбора дождевой и серой воды в муниципальную сеть, повторное использование, мониторинг качества и программы снижения потерь.  

### 2. Транспорт и логистика  
- **Мультимодальный транспортный узел

In [19]:
board.notes

[Note(id='905e2f', author_id='a9f900', author_role='Градостроительный планировщик', content='Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы (без количественных показателей)\n\n1. **Устойчивое развитие и экологическая безопасность**\n   - Формирование городской среды, ориентированной на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка муниципальных и частных организаций на экологически чистый транспорт.\n   - Увеличение доли зеленых насаждений и открытых пространств: создание новых парков, рекреационных зон вдоль реки Ижора, развитие линейных зеленых коридоров, интеграция вертикального озеленения в архитектуре.\n   - Внедрение принципов «зеленой» инфраструктуры в новых районах: энергоэффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.\n\n2. **Транспортная и логистическая интеграция**\n   - Развитие мультимодального транспортного 

In [20]:
from src import llm

res = llm.invoke([
    SystemMessage(f"""
    Ваша задача: синтезировать финальный ответ на поставленную задачу, основываясь ТОЛЬКО на материалах доски.
    ---
    Поставленная задача:
    `{board.question}`
    ---
    Записи на доске:
    {str(board.notes)}
    """)
])

In [23]:
board.notes

[Note(id='905e2f', author_id='a9f900', author_role='Градостроительный планировщик', content='Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы (без количественных показателей)\n\n1. **Устойчивое развитие и экологическая безопасность**\n   - Формирование городской среды, ориентированной на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка муниципальных и частных организаций на экологически чистый транспорт.\n   - Увеличение доли зеленых насаждений и открытых пространств: создание новых парков, рекреационных зон вдоль реки Ижора, развитие линейных зеленых коридоров, интеграция вертикального озеленения в архитектуре.\n   - Внедрение принципов «зеленой» инфраструктуры в новых районах: энергоэффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.\n\n2. **Транспортная и логистическая интеграция**\n   - Развитие мультимодального транспортного 

In [22]:
print(res.content)

**Целеполагание г. Гатчина (Ленинградская область) на 2026‑2035 годы**  
*(без привязки к количественным показателям; ориентировано на качественные результаты)*  

---

### 1. Устойчивое развитие и экологическая безопасность  
- Формировать городскую среду, ориентированную на снижение углеродного следа: развитие сети электромобилей, расширение инфраструктуры зарядных станций, поддержка перехода автопарка муниципальных и частных организаций на экологически чистый транспорт.  
- Увеличивать долю зелёных насаждений и открытых пространств: создание новых парков и рекреационных зон вдоль реки Ижора, развитие линейных зелёных коридоров, интеграция вертикального озеленения в архитектуру.  
- Внедрять принципы «зелёной» инфраструктуры в новых районах: энерго‑эффективные здания, системы сбора и повторного использования дождевой воды, умные системы уличного освещения.  

### 2. Транспортная и логистическая интеграция  
- Развивать мультимодальный транспортный узел, соединяющий железнодорожный во